<a href="https://colab.research.google.com/github/kookie-707/starzplay-internship/blob/main/Ai-movie-keywords/AI_Keywords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import openai
import os
import getpass
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output
import time
import io
from openai._exceptions import (RateLimitError, APIConnectionError, APIStatusError, OpenAIError)
import asyncio
import httpx

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
openai.api_key = os.environ.get("OPENAI_API_KEY")

In [ ]:
models = openai.models.list()
for m in models.data:
    print(m.id)

In [ ]:
file_path = input("Paste full CSV path: ")

try:
    df = pd.read_csv(file_path)
    print("Success")
    display(df.head())
except FileNotFoundError as e:
    print(f"Error message: {e}")


In [ ]:

prompt = f"""You are a metadata generation expert for a movie streaming platform. Given a movie or series title, respond with exactly 30 comma-separated keywords an actual user might type when searching.
First output 15 English keywords (the very first English keyword must be the title itself), then output 15 Arabic keywords (the very first Arabic keyword must be the title in Arabic).
Do not mix English and Arabic or insert any line breaks. The Arabic keywords should be treated separately and does not have to be a direct translation of the English keywords
Instructions:
- First English term → the exact title in English.
- Next 14 English terms → user-centric searches (themes, moods, settings, plot points, cast names, locations, and 2–3 similar titles, just the title text; do not prefix with “similar” or any label).
- After 15 English terms, immediately switch to Arabic:
- First Arabic term → the exact title in Arabic.
- Next 14 Arabic terms → user-centric searches (Arabic-language themes, moods, cast names in Arabic, locations in Arabic, and 2–3 popular Arabic titles, just the title text in Arabic; do not prefix with “مشابه” or any label).
- Include the title itself
- Focus on user search queries: themes, moods, setting, notable plot points avoid generic labels similar to “key characters” or “animated characters” or “popular characters.”
- The cast should include prominent actors or directors by name only (no labels such as “actor” or “director”).
- The locations can also be cultural context that a user might type (city names, country, or region).
- Also the similar Arabic titles should be for popular Arabic movies or series
- Do NOT overemphasize genres; only include them if they reflect how someone would search (e.g., “zombie movie” rather than “horror”).
- Avoid redundant and synonymous keywords.
- Do not mention any streaming platform names (eg: Netflix or HBO etc) or any channel names
- The keyword cannot be similar to or similar titles or similar shows or similar movies or notable moments or any other label
- For the similar titles just output the title's names only
- **Output must be exactly one line with 30 comma-separated terms.**

Return exactly one comma-separated list with no explanations, no numbering, no sudden line breaks and no extra formatting."""

fallback_prompt = """You are a metadata-generation expert. Given a movie or series title,
respond with 4–6 comma-separated genres: first the English title, then 2–3 English genres, then the title in Arabic and then 2–3 Arabic genres.
All on one line, no newlines, no labels, no extra text."""

def generate_keywords(title, max_retries=5, retry_delay=8):
   def call_api(prompt_text):
        return openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt_text.strip()},
                {"role": "user",   "content": title}
            ],
            max_tokens=200,
            temperature=0.7,
            timeout=20).choices[0].message.content.strip()

   for attempt in range(max_retries):
        try:
            output = call_api(prompt)
            terms = [t.strip() for t in output.split(",") if t.strip()]
            if len(terms) >= 25 and not output.endswith(","):
              return ", ".join(terms)
            time.sleep(retry_delay)

        except (RateLimitError, APIConnectionError, APIStatusError, OpenAIError) as e:
            print(f"[Attempt {attempt + 1}] OpenAI error for '{title}': {e}")
            time.sleep(retry_delay)

        except httpx.TimeoutException as e:
            print(f"[Attempt {attempt + 1}] Timeout error for '{title}': {e}")
            time.sleep(retry_delay)

        except Exception as e:
            print(f"[Attempt {attempt + 1}] Unexpected error for '{title}': {e}")
            time.sleep(retry_delay)
   try:
        fallback_output = call_api(fallback_prompt)
        fallback_terms = [t.strip() for t in fallback_output.split(",") if t.strip()]
        if len(fallback_terms) >= 4 and not fallback_output.endswith(","):
            return ", ".join(fallback_terms)
   except Exception as e:
        print(f"[Fallback genre] Error for '{title}': {e}")

   return f"{title}"




In [ ]:
for title in df['Video Title']:
    print(title)
    print(generate_keywords(title))

In [ ]:
save_path = input("Paste full CSV save path: ")
def run_generation():
    results = []
    for title in tqdm(df['Video Title'], desc="Generating keywords"):
        result = generate_keywords(title)
        results.append(result)
        time.sleep(1)
    df['AI Keywords'] = results
    display(df.head())

    df.to_csv(save_path, index=False)
    print(f"Success: Saved to {save_path}")

generate_button = widgets.Button(description="Generate")
display(generate_button)

def on_generate_button_click(_):
    run_generation()

generate_button.on_click(on_generate_button_click)
